# Lesson 2 · NumPy Superpowers: Vectorization, Broadcasting, Matrices & Randomness
### Stop writing loops. Start thinking in whole arrays.

| | |
|---|---|
| **Type** | SOLUTIONS 02 |
| **Time** | about 120 minutes |
| **Level** | Beginner → Intermediate |
| **Prerequisites** | Lesson 1 (arrays, indexing, slicing, masks) |

**By the end of this notebook you will be able to:**

- Replace Python loops with vectorised operations and *measure* the gain
- Summarise data with aggregations along any axis
- Predict the result (or the error!) of any broadcasting operation using the two shape rules
- Use matrix multiplication, inverses and linear solvers on real questions
- Handle floating-point quirks, NaNs, differences and percentage changes
- Generate reproducible random numbers, run a Monte-Carlo simulation and bootstrap an average

## 🛠️ Setup (identical to the lesson notebook)

In [1]:

import sys; sys.path.insert(0, "..")
import time
import numpy as np
import pandas as pd
from course_utils import DATA_DIR, check

rng = np.random.default_rng(7)
# 8 weeks x 3 crops of weekly trading volume (tonnes): maize, beans, tomatoes
weekly_volume = np.round(rng.normal([50, 120, 30], [5, 15, 4], size=(8, 3)), 1)
# Maize price (KES/kg): 3 markets x 4 weeks, and the KES-per-USD exchange rate in each of those weeks
price_kes = np.array([[48.0, 49.5, 51.0, 50.2],
                      [52.0, 53.5, 55.0, 54.1],
                      [45.5, 46.0, 47.2, 46.8]])
usd_rate = np.array([128.5, 129.0, 130.2, 131.0])
# A short price series
series = np.array([50, 52, 51, 55, 54, 60, 57, 63, 61, 66])
# Real weekly maize price & volume at Wakulima Market
df = pd.read_csv(DATA_DIR / "market_prices_clean.csv")
_m = (df["crop"] == "Maize") & (df["market"] == "Wakulima Market")
mz_price = df.loc[_m, "price_kes_per_kg"].to_numpy()
mz_vol = df.loc[_m, "volume_kg"].to_numpy()
print("Setup complete ✔")


Setup complete ✔


### 🧩 Problem 2.1: Standardise the volumes

Standardise each **column** of `weekly_volume` to a z-score: `(x − column mean) / column standard deviation` (use NumPy's default `std`). Return the **first row** of the standardised matrix, rounded to 2 decimals.

*Expected type:* an array of 3 numbers.

In [2]:
z = (weekly_volume - weekly_volume.mean(axis=0)) / weekly_volume.std(axis=0)
answer = np.round(z[0], 2)

check('02-p1', answer)

✅ 02-p1: correct! Nicely done.


True

### 🧩 Problem 2.2: Convert to dollars, week by week

`price_kes` has shape `(3, 4)` and `usd_rate` has shape `(4,)` (KES per USD in each week). Return the USD price matrix (KES price ÷ rate), rounded to **3 decimals**.

*Expected type:* a `(3, 4)` array.

In [3]:
answer = np.round(price_kes / usd_rate, 3)

check('02-p2', answer)

✅ 02-p2: correct! Nicely done.


True

### 🧩 Problem 2.3: The biggest weekly jump

For the array `series`, compute the **percentage change** from each week to the next and return the **largest** one, rounded to 2 decimals.

*Expected type:* float (a percentage such as `7.84`).

In [4]:
answer = round(float((np.diff(series) / series[:-1] * 100).max()), 2)

check('02-p3', answer)

✅ 02-p3: correct! Nicely done.


True

### 🧩 Problem 2.4: Which prices?

A trader pays **5 kg of maize + 2 kg of beans = 600 KES**, and another pays **3 kg of maize + 4 kg of beans = 780 KES**. Use a NumPy linear solver to find the per-kg prices.

*Expected type:* an array `[maize_price, beans_price]`.

In [5]:
A = np.array([[5, 2],
              [3, 4]])
b = np.array([600, 780])
answer = np.linalg.solve(A, b)

check('02-p4', answer)

✅ 02-p4: correct! Nicely done.


True

### 🧩 Problem 2.5: Tight-market weeks (no loops!)

`mz_price` and `mz_vol` hold weekly maize price and volume at Wakulima. A week is **tight** if its price is **above the median price** *and* its volume is **below the median volume**. How many tight weeks are there?

*Expected type:* an integer.

In [6]:
tight = (mz_price > np.median(mz_price)) & (mz_vol < np.median(mz_vol))
answer = int(tight.sum())

check('02-p5', answer)

✅ 02-p5: correct! Nicely done.


True

### 🧩 Problem 2.6: Will the price spike? (Monte-Carlo)

A price starts at 50 and each of the next 12 weeks adds a random step drawn from Normal(mean 0, sd 2). Using `rng = np.random.default_rng(7)` and **exactly** `steps = rng.normal(0, 2, size=(10000, 12))`, simulate 10,000 futures. Return the **proportion** whose final price (50 + sum of steps) is **above 55**.

*Expected type:* float between 0 and 1.

In [7]:
rng = np.random.default_rng(7)
steps = rng.normal(0, 2, size=(10000, 12))
final = 50 + steps.sum(axis=1)
answer = float((final > 55).mean())

check('02-p6', answer)

✅ 02-p6: correct! Nicely done.


True

### 🧩 Problem 2.7: Revenue by market

Using `prices_tbl` (3 markets × 3 crops, defined in section 4; recreate it: rows `[60,150,90]`, `[65,160,95]`, `[58,145,85]`) and the quantities `kg_sold = [1000, 400, 700]`, compute the revenue at each market with a **single matrix product**. Then return the revenue of the **best** market minus the revenue of the **worst** market.

*Expected type:* a number.

In [8]:
prices_tbl = np.array([[60, 150, 90], [65, 160, 95], [58, 145, 85]])
kg_sold = np.array([1000, 400, 700])
rev = prices_tbl @ kg_sold
answer = int(rev.max() - rev.min())

check('02-p7', answer)

✅ 02-p7: correct! Nicely done.


True

### ✍️ Problem 2.8: Explain the broadcast

In your own words (3–4 sentences): why does `price_kes + np.array([1.0, 2.5, 0.5])` fail, and what are **two different ways** to fix it depending on whether the three numbers are per-market or per-week adjustments?

**Model answer**

`price_kes` has shape `(3, 4)`; the vector has shape `(3,)`. Comparing from the right, `4 ≠ 3` and neither is 1, so broadcasting fails. If the three numbers are **per-market** (one per row), make them a column: `v[:, np.newaxis]` or `v.reshape(3, 1)`. If they were **per-week**, they would need length 4 and would work directly as a `(4,)` row.